In [0]:
# ============================================================
# Silver — Source 04: MSK Kafka Clickstream
#
# Transformations:
#   - Cast event_ts ISO string to timestamp
#   - Normalise event_type, device, browser, traffic_source
#   - Handle anonymous sessions (user_id null = valid)
#   - Reject null event_id or session_id → quarantine
#   - Deduplicate on event_id
#
# Source:  bronze.src_04_clickstream.events
# Target:  silver.src_04_clickstream.events
# Quarantine: silver.quarantine.src_04_clickstream
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.window import Window

BRONZE_CATALOG = 'bronze'
SILVER_CATALOG = 'silver'
TARGET_TABLE = f'{SILVER_CATALOG}.src_04_clickstream.events'
QUARANTINE_TABLE = f'{SILVER_CATALOG}.quarantine.src_04_clickstream'

VALID_EVENT_TYPES = ['page_view', 'product_view', 'add_to_cart', 'remove_from_cart',
                     'begin_checkout', 'purchase', 'view_cart', 'search', 'login', 'logout']

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {SILVER_CATALOG}.src_04_clickstream')
print('Silver Source 04 Clickstream — starting...')


In [0]:
# ── LOAD AND CLEAN ────────────────────────────────────────────
bronze = spark.table(f'{BRONZE_CATALOG}.src_04_clickstream.events')
total = bronze.count()
print(f'Bronze rows: {total}')

# Step 1: Cast timestamp
df = bronze.withColumn('event_ts', F.to_timestamp(F.col('event_ts')))

# Step 2: Normalise string fields
df = df \
    .withColumn('event_type',     F.lower(F.trim(F.col('event_type')))) \
    .withColumn('device',         F.lower(F.trim(F.col('device')))) \
    .withColumn('browser',        F.lower(F.trim(F.col('browser')))) \
    .withColumn('traffic_source', F.lower(F.trim(F.col('traffic_source')))) \
    .withColumn('country',        F.upper(F.trim(F.col('country')))) \
    .withColumn('product_sku',    F.upper(F.trim(F.col('product_sku'))))

# Step 3: Flag bad rows
# NOTE: user_id null is VALID — 30% anonymous sessions expected
bad = df.filter(
    F.col('event_id').isNull() |
    F.col('session_id').isNull() |
    F.col('event_ts').isNull() |
    ~F.col('event_type').isin(VALID_EVENT_TYPES)
).withColumn('quarantine_reason', F.lit('failed_validation')) \
 .withColumn('source_table', F.lit('events'))

# Step 4: Good rows — dedup on event_id
good = df.filter(
    F.col('event_id').isNotNull() &
    F.col('session_id').isNotNull() &
    F.col('event_ts').isNotNull() &
    F.col('event_type').isin(VALID_EVENT_TYPES)
)

w = Window.partitionBy('event_id').orderBy(F.col('event_ts').desc())
good = good.withColumn('_rn', F.row_number().over(w)) \
           .filter(F.col('_rn') == 1).drop('_rn')

bad_count = bad.count()
good_count = good.count()
anon_count = good.filter(F.col('user_id').isNull()).count()

print(f'Clickstream: {total} total → {good_count} clean, {bad_count} quarantined ({bad_count/total*100:.1f}%)')
print(f'Anonymous sessions: {anon_count}/{good_count} ({anon_count/good_count*100:.1f}%)')

# Step 5: Write to Silver
if spark.catalog.tableExists(TARGET_TABLE):
    dt = DeltaTable.forName(spark, TARGET_TABLE)
    dt.alias('t').merge(good.alias('s'), 't.event_id = s.event_id') \
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print('MERGE complete')
else:
    good.write.format('delta').mode('overwrite').saveAsTable(TARGET_TABLE)
    print('Initial load complete')

# Step 6: Quarantine
if bad_count > 0:
    quarantine = bad.select(
        F.lit('src_04_clickstream').alias('source'),
        F.col('source_table'),
        F.col('quarantine_reason'),
        F.current_timestamp().alias('quarantined_at'),
        F.to_json(F.struct(*[c for c in bad.columns if c not in ['quarantine_reason','source_table']])).alias('raw_record')
    )
    quarantine.write.format('delta').mode('append') \
        .option('mergeSchema', 'true').saveAsTable(QUARANTINE_TABLE)
    print(f'✅ {bad_count} rows quarantined')
else:
    print('No quarantine rows')


In [0]:
# ── VERIFY ────────────────────────────────────────────────────
count = spark.sql(f'SELECT COUNT(*) as cnt FROM {TARGET_TABLE}').collect()[0]['cnt']
print(f'silver.src_04_clickstream.events: {count} rows')
spark.sql(f"""
    SELECT event_type, COUNT(*) as cnt 
    FROM {TARGET_TABLE} 
    GROUP BY event_type 
    ORDER BY cnt DESC
""").show()
